# RSICD Adapter-CLIP — Session 5 of 5 — Ablations + Figures

Auto-generated from the rsicd-clip-adapter repo. Source of truth: `KAGGLE_RUNBOOK.md`.

**Before running this notebook:**
1. **Data** — pick ONE of these:
   - **Option A** (recommended): Click **+ Add data** in the right panel, search `rsicd-image-caption-dataset`, click **Add**
   - **Option B** (if you don't want to attach a Dataset): Download `train.csv`, `valid.csv`, `test.csv` from the thedevastator Kaggle dataset page, then in the Files panel (right), click **Upload** and drop the 3 files. They land in `/kaggle/working/`.
2. **Previous-session Datasets** (sessions 2-5 only): click **+ Add data** → add any `rsicd-adapter-s*` or `rsicd-fullfinetune` Datasets you saved earlier
3. Settings: **Accelerator = GPU P100 or T4**, **Internet = ON**
4. Click **Save Version → Save Output** at the end of this session

Cell 2 will auto-detect which option you used and report back.

In [ ]:
# === Install + clone + env ===
# IMPORTANT: always cd to /kaggle/working (an absolute path) before
# cloning, so re-running this cell doesn't nest rsicd-clip-adapter
# inside itself.
%cd /kaggle/working
!rm -rf rsicd-clip-adapter
!pip install open_clip_torch faiss-cpu ftfy accelerate pyyaml -q
!git clone https://github.com/Vatsal057/rsicd-clip-adapter.git
%cd /kaggle/working/rsicd-clip-adapter
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
print("Repo cloned, deps installed.")
print(f"CWD: {os.getcwd()}")

In [ ]:
# === Sanity: GPU + dataset location ===
import torch, os
from pathlib import Path
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
print()

# The data can come from two places:
#   (A) Attached Kaggle Dataset: /kaggle/input/<slug>/train.csv
#   (B) Direct upload to /kaggle/working/ (or any subdir thereof)
data_path = None

# (A) Try the standard /kaggle/input/ dataset slugs
if Path('/kaggle/input').exists():
    for p in sorted(Path('/kaggle/input').iterdir()):
        if p.is_dir() and (p / 'train.csv').exists():
            data_path = p
            break

# (B) Try direct upload locations
if data_path is None and Path('/kaggle/working').exists():
    for candidate in [
        Path('/kaggle/working'),
        Path('/kaggle/working/rsicd-clip-adapter'),
    ]:
        if (candidate / 'train.csv').exists():
            data_path = candidate
            break
    # Last resort: walk /kaggle/working/ recursively and find train.csv
    if data_path is None:
        try:
            for csv in Path('/kaggle/working').rglob('train.csv'):
                data_path = csv.parent
                break
        except Exception:
            pass

# Report what we found
if data_path is None:
    print("ERROR: Could not find an RSICD dataset.")
    print()
    print("Option A — Attach as Kaggle Dataset (preferred):")
    print("  1. Click '+ Add data' in the right panel")
    print("  2. Search 'rsicd-image-caption-dataset' (the thedevastator version)")
    print("  3. Click 'Add'")
    print()
    print("Option B — Upload CSVs directly (faster, but you re-upload every session):")
    print("  1. Download train.csv, valid.csv, test.csv from the thedevastator")
    print("     Kaggle dataset page")
    print("  2. In the Files panel (right), click 'Upload' and drop the 3 CSVs")
    print("  3. They land in /kaggle/working/")
    print()
    print("Currently attached under /kaggle/input/:")
    try:
        for name in sorted(p.name for p in Path('/kaggle/input').iterdir()):
            print(f"   - {name}")
    except FileNotFoundError:
        print("   (no /kaggle/input/ directory)")
    print()
    print("Currently in /kaggle/working/:")
    try:
        for name in sorted(p.name for p in Path('/kaggle/working').iterdir()):
            print(f"   - {name}")
    except FileNotFoundError:
        print("   (no /kaggle/working/ directory)")
    # Set a sentinel so cell 3 (conversion) can detect this and skip
    DATA_OK = False
else:
    os.environ['RSICD_ARCHIVE'] = str(data_path)
    print(f"Using dataset: {data_path}")
    csvs = sorted(p.name for p in data_path.glob('*.csv'))
    print(f"  CSVs:   {csvs}")
    DATA_OK = True

## Required: attach the previous sessions' Datasets
Click **+ Add data** → search and add:
- `rsicd-adapter-s3` (for the adapter checkpoint used by figures)
- `rsicd-fullfinetune` (for the full-fine-tune metrics used by figures)

In [ ]:
# === Restore checkpoint from previous session ===
import shutil, os
from pathlib import Path
src = Path("/kaggle/input/rsicd-adapter-s3/adapter_best.pt")
if not src.exists():
    print(f"WARN: {src} not found. Did you forget to '+ Add data' rsicd-adapter-s3?")
else:
    Path("results/checkpoints").mkdir(parents=True, exist_ok=True)
    shutil.copy(src, "results/checkpoints/adapter_best.pt")
    print(f"Restored: {src}")
    # Also restore training history if it was saved
    src_h = Path("/kaggle/input/rsicd-adapter-s3/training_history_adapter.json")
    if src_h.exists():
        Path("results/metrics").mkdir(parents=True, exist_ok=True)
        shutil.copy(src_h, "results/metrics/training_history_adapter.json")
        print("Restored training history.")

If you want to split this session, run cells one at a time. Each ablation writes its own JSON when it finishes, so a mid-run disconnect loses at most one ablation.

In [ ]:
# === Ablations (placement + hidden_dim + data_size + residual) ===
# This runs ~20 hours on P100. If you hit the 12 h Kaggle limit, save what you have
# and resume in another notebook. Each ablation writes its own JSON when it finishes.
!python scripts/05_ablations.py

In [ ]:
# === Regenerate 4 paper figures with real data ===
!python scripts/06_qualitative.py

In [ ]:
# === Package output for next session ===
# After this cell, click 'Save Version' (top right) with 'Save Output' enabled.
# Then go to the Output tab -> 'New Dataset' -> name it 'rsicd-final'.
# The next session will attach this Dataset via '+ Add data'.
import shutil, os
out_dir = "/kaggle/working/rsicd-final"
os.makedirs(out_dir, exist_ok=True)
src = "results"
if os.path.exists(src):
    dst = os.path.join(out_dir, os.path.basename(src)) if not os.path.isdir(src) else out_dir
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(out_dir, os.path.basename(src)), dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
    print(f"  + {src} -> {out_dir}")
src = "paper/figures"
if os.path.exists(src):
    dst = os.path.join(out_dir, os.path.basename(src)) if not os.path.isdir(src) else out_dir
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(out_dir, os.path.basename(src)), dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
    print(f"  + {src} -> {out_dir}")
src = "paper/main.tex"
if os.path.exists(src):
    dst = os.path.join(out_dir, os.path.basename(src)) if not os.path.isdir(src) else out_dir
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(out_dir, os.path.basename(src)), dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
    print(f"  + {src} -> {out_dir}")
src = "paper/refs.bib"
if os.path.exists(src):
    dst = os.path.join(out_dir, os.path.basename(src)) if not os.path.isdir(src) else out_dir
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(out_dir, os.path.basename(src)), dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
    print(f"  + {src} -> {out_dir}")
print(f"\nReady. Save this notebook version with 'Save Output' ON, then convert output to Dataset 'rsicd-final'.")

## Done!
Save the version, then convert output to Dataset `rsicd-final`.

**This is the last session.** Download `rsicd-final` from Kaggle → merge into your local repo → fill in `[TBD]` in `main.tex` → push to GitHub.